# Pré-processamento 9 — Revisão de compatibilidade entre palavra e categoria
## Decisões aprovadas
Aplicar 10 correções do grupo Verbo Principal Gerúndio e 14 do grupo Nome Comun Masculino Plural e 7 do grupo Nome Comun Feminino Plural e 5 do grupo Verbo Principal Particípio Singular Masculino e 5 do grupo Verbo Principal Indicativo Passado Primeira Pessoa Singular (41 aprovadas ao todo). Não alterar LEMA nem grafia.
O código aceita categorias já corrigidas, permitindo reexecução. Valida palavras pela identidade e pelo lema, não pela antiga posição da linha.
Para acrescentar remoções, edite a lista e reexecute desde o carregamento.
O salvamento mostra as contagens da execução e o total final da base. Arquivos intermediários são criados em pasta temporária e limpos automaticamente ao concluir ou ocorrer uma exceção. As versões anteriores são mantidas pelo Git.


In [21]:
from pathlib import Path
import json, hashlib, os, subprocess
import pandas as pd
from IPython.display import display
arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
base_original = base.copy(deep=True)
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
print("Registros carregados:", len(base))


Registros carregados: 7774


## 1. Função de correção de categoria e preparação do resultado

In [22]:
# Função compartilhada pelos grupos; todo o código de correção está neste notebook.
# Altera exclusivamente CATMORF, sem atribuir valores em LEMA.
def aplicar_correcoes_categoria(resultado, correcoes):
    antes = resultado.copy(deep=True)
    aplicadas = []
    for item in correcoes:
        mascara = resultado["PALAVRA"].eq(item["PALAVRA"])
        assert mascara.sum() == 1, f"Palavra ausente ou duplicada: {item['PALAVRA']}"
        indice = resultado.index[mascara][0]
        atual = resultado.at[indice, "CATMORF"]
        assert atual in (item["CATMORF"], item["CATEGORIA_SUGERIDA"]), f"Categoria mudou: {item['PALAVRA']}"
        assert resultado.at[indice, "LEMA"] == item["LEMA"], f"Lema mudou: {item['PALAVRA']}"
        if atual != item["CATEGORIA_SUGERIDA"]:
            resultado.at[indice, "CATMORF"] = item["CATEGORIA_SUGERIDA"]
            aplicadas.append({**item, "CATEGORIA_ANTERIOR": atual})
    pd.testing.assert_frame_equal(antes.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
    return aplicadas

resultado = base.copy(deep=True)
alteracoes = []
correcoes_aprovadas = []


## 1.1. Verbo Principal Gerúndio — aplicar 10 correções aprovadas

In [3]:
# Lista completa das decisões aprovadas — Verbo Principal Gerúndio
correcoes_gerundio = [{'LINHA_EXCEL': 233,
  'PALAVRA': 'ajeitei',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 239,
  'PALAVRA': 'ajudar',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 296,
  'PALAVRA': 'almoçou',
  'LEMA': 'almoçar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de almoçar, terceira pessoa do singular; o gerúndio é '
                   'almoçando.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 322,
  'PALAVRA': 'amadurecer',
  'LEMA': 'amaciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 346,
  'PALAVRA': 'amassei',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 382,
  'PALAVRA': 'andarem',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Terceira Pessoa Plural',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal na terceira pessoa do plural ou futuro do subjuntivo '
                   'na mesma pessoa. A proposta de infinitivo é uma opção válida, sem preferência '
                   'obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 383,
  'PALAVRA': 'andares',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Segunda Pessoa Singular',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal ou futuro do subjuntivo na segunda pessoa do '
                   'singular; também pode ser plural do substantivo andar. A proposta verbal é uma '
                   'opção válida, sem preferência obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 426,
  'PALAVRA': 'ao',
  'LEMA': 'anunciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Preposição + Artigo Definido Masculino Singular',
  'JUSTIFICATIVA': 'Contração de a com o, compatível com o rótulo já existente na base. Não é '
                   'gerúndio de anunciar indicado no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 902,
  'PALAVRA': 'bem-vindo',
  'LEMA': 'bem-vindo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Adjetivo masculino singular; também pode ser usado em expressão de saudação. A '
                   'terminação vindo não torna o composto um gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4676,
  'PALAVRA': 'marimbondo',
  'LEMA': 'marimbondo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa o inseto; terminar em ndo não '
                   'caracteriza gerúndio por si só.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_gerundio)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_gerundio)
print("Grupo: Verbo Principal Gerúndio")
print("Correções aprovadas:", len(correcoes_gerundio))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_gerundio) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_gerundio)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Gerúndio
Correções aprovadas: 10
Aplicadas nesta execução: 0
Já estavam corrigidas: 10


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajeitei,ajeitar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
ajudar,ajudar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
almoçou,almoçar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
amadurecer,amaciar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
amassei,amassar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
andarem,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Terceira Pessoa Plural,Aprovada
andares,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Segunda Pessoa Singular,Aprovada
ao,anunciar,Verbo Principal Gerúndio,Preposição + Artigo Definido Masculino Singular,Aprovada
bem-vindo,bem-vindo,Verbo Principal Gerúndio,Adjetivo Qualificativo Masculino Singular,Aprovada
marimbondo,marimbondo,Verbo Principal Gerúndio,Nome Comun Masculino Singular,Aprovada


## 1.2. Nome Comun Masculino Plural — aplicar 14 correções aprovadas


In [4]:
# Lista completa das decisões aprovadas — Nome Comun Masculino Plural
correcoes_masculino_plural = [{'LINHA_EXCEL': 228,
  'PALAVRA': 'ainda',
  'LEMA': 'aimeudeus',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Advérbio com valores temporais, de continuidade ou inclusão; não é substantivo '
                   'masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 254,
  'PALAVRA': 'alcança',
  'LEMA': 'albuns',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Admite presente do indicativo de alcançar ou imperativo. A proposta verbal é '
                   'uma opção válida; não corresponde a nome masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 311,
  'PALAVRA': 'alça',
  'LEMA': 'albuns',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular ou forma verbal de alçar; a proposta '
                   'nominal é uma opção válida, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 315,
  'PALAVRA': 'alípio',
  'LEMA': 'alimento',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 358,
  'PALAVRA': 'americanizado',
  'LEMA': 'amendoim',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Particípio masculino singular de americanizar; também admite adjetivo. Mesmo '
                   'na leitura substantivada, a forma é singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 365,
  'PALAVRA': 'amizades',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; corrigir apenas o gênero da categoria.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 385,
  'PALAVRA': 'ande',
  'LEMA': 'andar',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de andar no presente do subjuntivo; primeira pessoa do singular ou '
                   'imperativo também são possíveis. A análise proposta não tem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 393,
  'PALAVRA': 'ângelo',
  'LEMA': 'anel',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 404,
  'PALAVRA': 'anotar',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de anotar. Uma eventual leitura nominal do infinitivo seria '
                   'singular; não é nome masculino plural na forma cadastrada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 406,
  'PALAVRA': 'ansiosa',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Forma feminina singular de ansioso. Mesmo em leitura substantivada, o gênero e '
                   'o número da categoria atual não correspondem.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2450,
  'PALAVRA': 'dicas',
  'LEMA': 'dica',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; corrigir apenas o gênero da categoria.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6526,
  'PALAVRA': 'rubens',
  'LEMA': 'rubens',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7135,
  'PALAVRA': 'teus',
  'LEMA': 'teus',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Determinante Possessivo Segunda Pessoa Masculino Plural Singular',
  'JUSTIFICATIVA': 'Possessivo da segunda pessoa: masculino plural para o elemento possuído, '
                   'possuidor singular. Também admite análise pronominal. O rótulo proposto é novo '
                   'na base e segue o padrão de teu e meus, sem alterar LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7246,
  'PALAVRA': 'torres',
  'LEMA': 'torr',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Plural feminino de torre; também admite sobrenome ou segunda pessoa do '
                   'presente de torrar. A proposta nominal feminina é uma análise possível.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_masculino_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_masculino_plural)
print("Grupo: Nome Comun Masculino Plural")
print("Correções aprovadas:", len(correcoes_masculino_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_masculino_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_masculino_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Masculino Plural
Correções aprovadas: 14
Aplicadas nesta execução: 0
Já estavam corrigidas: 14


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ainda,aimeudeus,Nome Comun Masculino Plural,Advérbio Geral,Aprovada
alcança,albuns,Nome Comun Masculino Plural,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
alça,albuns,Nome Comun Masculino Plural,Nome Comun Feminino Singular,Aprovada
alípio,alimento,Nome Comun Masculino Plural,Nome Própio Pessoa,Aprovada
americanizado,amendoim,Nome Comun Masculino Plural,Verbo Principal Particípio Singular Masculino,Aprovada
amizades,amigo,Nome Comun Masculino Plural,Nome Comun Feminino Plural,Aprovada
ande,andar,Nome Comun Masculino Plural,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
ângelo,anel,Nome Comun Masculino Plural,Nome Própio Pessoa,Aprovada
anotar,ano,Nome Comun Masculino Plural,Verbo Principal Infinitivo,Aprovada
ansiosa,ano,Nome Comun Masculino Plural,Adjetivo Qualificativo Feminino Singular,Aprovada


## 1.3. Nome Comun Feminino Plural — aplicar 7 correções aprovadas


In [5]:
# Lista completa das decisões aprovadas — Nome Comun Feminino Plural
correcoes_feminino_plural = [{'LINHA_EXCEL': 260,
  'PALAVRA': 'álcool',
  'LEMA': 'alça',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; gênero e número não correspondem à categoria '
                   'feminina plural. Preservar o lema cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 363,
  'PALAVRA': 'amigos',
  'LEMA': 'amígdala',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural masculino de amigo; corrigir o gênero da categoria, sem alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 368,
  'PALAVRA': 'amora',
  'LEMA': 'amizade',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular, nome da fruta; também pode ser forma de '
                   'amorar. A proposta nominal é uma análise possível; a forma não é plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 419,
  'PALAVRA': 'antídoto',
  'LEMA': 'antena',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; gênero e número não correspondem à categoria '
                   'feminina plural. Preservar o lema cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2055,
  'PALAVRA': 'crianções',
  'LEMA': 'criança',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural de crianção, forma aumentativa masculina. Corrigir o gênero da '
                   'categoria, preservando o lema criança e a grafia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6183,
  'PALAVRA': 'quebra-cabeças',
  'LEMA': 'quebra-cabeças',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'O composto é substantivo masculino de dois números. A leitura plural é '
                   'possível; corrigir apenas o gênero, sem impor preferência entre singular e '
                   'plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6621,
  'PALAVRA': 'salvá-las',
  'LEMA': 'salvá-las',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de salvar com pronome enclítico las. O gênero e o número do pronome '
                   'não tornam a expressão um substantivo feminino plural.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_feminino_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_feminino_plural)
print("Grupo: Nome Comun Feminino Plural")
print("Correções aprovadas:", len(correcoes_feminino_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_feminino_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_feminino_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Feminino Plural
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
álcool,alça,Nome Comun Feminino Plural,Nome Comun Masculino Singular,Aprovada
amigos,amígdala,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
amora,amizade,Nome Comun Feminino Plural,Nome Comun Feminino Singular,Aprovada
antídoto,antena,Nome Comun Feminino Plural,Nome Comun Masculino Singular,Aprovada
crianções,criança,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
quebra-cabeças,quebra-cabeças,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
salvá-las,salvá-las,Nome Comun Feminino Plural,Verbo Principal Infinitivo,Aprovada


## 1.4. Verbo Principal Particípio Singular Masculino — aplicar 5 correções aprovadas


In [6]:
# Lista completa das decisões aprovadas — Verbo Principal Particípio Singular Masculino
correcoes_participio_masculino = [{'LINHA_EXCEL': 250,
  'PALAVRA': 'alaranjadas',
  'LEMA': 'alar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Plural Feminino',
  'JUSTIFICATIVA': 'Particípio feminino plural de alaranjar; também admite adjetivo. A leitura de '
                   'particípio pode ser mantida, corrigindo gênero e número, sem alterar o lema '
                   'cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 336,
  'PALAVRA': 'amarre',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Presente do subjuntivo de amarrar; primeira pessoa do singular ou imperativo '
                   'também são possíveis. A proposta é uma análise válida, sem preferência '
                   'obrigatória; não é particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 345,
  'PALAVRA': 'amassasse',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Imperfeito Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito imperfeito do subjuntivo de amassar. Primeira e terceira pessoas do '
                   'singular são possíveis; a proposta não estabelece preferência. Não é '
                   'particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 397,
  'PALAVRA': 'animal',
  'LEMA': 'animar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular; também pode ser adjetivo de dois '
                   'gêneros. Não é particípio de animar, cujo particípio é animado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 412,
  'PALAVRA': 'anterior',
  'LEMA': 'antecipar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Adjetivo de dois gêneros no singular, para aquilo que vem antes. Não é '
                   'particípio de antecipar, cujo particípio é antecipado.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_participio_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_participio_masculino)
print("Grupo: Verbo Principal Particípio Singular Masculino")
print("Correções aprovadas:", len(correcoes_participio_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_participio_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_participio_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Particípio Singular Masculino
Correções aprovadas: 5
Aplicadas nesta execução: 0
Já estavam corrigidas: 5


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
alaranjadas,alar,Verbo Principal Particípio Singular Masculino,Verbo Principal Particípio Plural Feminino,Aprovada
amarre,amarrar,Verbo Principal Particípio Singular Masculino,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
amassasse,amassar,Verbo Principal Particípio Singular Masculino,Verbo Principal Subjuntivo Imperfeito Terceira Pessoa Singular,Aprovada
animal,animar,Verbo Principal Particípio Singular Masculino,Nome Comun Masculino Singular,Aprovada
anterior,antecipar,Verbo Principal Particípio Singular Masculino,Adjetivo Qualificativo Comum Singular,Aprovada


## 1.5. Verbo Principal Indicativo Passado Primeira Pessoa Singular — aplicar 5 correções aprovadas


In [23]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Passado Primeira Pessoa Singular
correcoes_passado_primeira = [{'LINHA_EXCEL': 245,
  'PALAVRA': 'ajudou',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de ajudar na terceira pessoa do singular; a primeira pessoa '
                   'é ajudei.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 348,
  'PALAVRA': 'ambiente',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular, além de adjetivo. A proposta nominal é '
                   'uma opção válida; não corresponde a forma do passado de amassar indicada no '
                   'lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 389,
  'PALAVRA': 'andreia',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Andreia, preservando a grafia cadastrada; não '
                   'corresponde à primeira pessoa do passado de andar indicada no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2418,
  'PALAVRA': 'detetive',
  'LEMA': 'detetive',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo que designa investigador; a leitura masculina singular é possível, '
                   'sem preferência obrigatória sobre outras análises nominais compatíveis. Não é '
                   'forma de passado na primeira pessoa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5779,
  'PALAVRA': 'piuí',
  'LEMA': 'piuí',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino que designa aves chamadas piuí. A proposta segue '
                   'essa leitura lexical; se a ocorrência for apenas vocalização, reavaliar a '
                   'categoria com contexto.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_passado_primeira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_passado_primeira)
print("Grupo: Verbo Principal Indicativo Passado Primeira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_passado_primeira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_passado_primeira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_passado_primeira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Passado Primeira Pessoa Singular
Correções aprovadas: 5
Aplicadas nesta execução: 0
Já estavam corrigidas: 5


,PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
0,ajudou,ajudar,Verbo Principal Indicativo Passado Primeira Pe...,Verbo Principal Indicativo Passado Terceira Pe...,Aprovada
1,ambiente,amassar,Verbo Principal Indicativo Passado Primeira Pe...,Nome Comun Masculino Singular,Aprovada
2,andreia,andar,Verbo Principal Indicativo Passado Primeira Pe...,Nome Própio Pessoa,Aprovada
3,detetive,detetive,Verbo Principal Indicativo Passado Primeira Pe...,Nome Comun Masculino Singular,Aprovada
4,piuí,piuí,Verbo Principal Indicativo Passado Primeira Pe...,Nome Comun Masculino Singular,Aprovada


## 2. Palavras identificadas como fora do escopo

In [8]:
palavras_remover = ['vamuos', 'ohmeudeus']

total_antes = len(resultado)
mascara_remocao = resultado["PALAVRA"].isin(palavras_remover)
removidas = resultado.loc[mascara_remocao].copy()
nao_encontradas = sorted(set(palavras_remover) - set(removidas["PALAVRA"]))
resultado = resultado.loc[~mascara_remocao].reset_index(drop=True)
quantidade_removida = len(removidas)
assert total_antes - quantidade_removida == len(resultado)
print("Remoções previstas nesta execução:", quantidade_removida)
print("Total previsto na base:", len(resultado))
if nao_encontradas:
    print("Palavras já removidas ou não encontradas:", nao_encontradas)
display(removidas[["PALAVRA", "LEMA", "CATMORF"]])


Remoções previstas nesta execução: 0
Total previsto na base: 7774
Palavras já removidas ou não encontradas: ['ohmeudeus', 'vamuos']


PALAVRA,LEMA,CATMORF


## 3. Salvamento e contagens

In [24]:
assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada, "A fonte mudou durante a execução."
retidas = base_original.loc[~base_original["PALAVRA"].isin(palavras_remover)].reset_index(drop=True)
pd.testing.assert_frame_equal(retidas.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
from tempfile import TemporaryDirectory
# A limpeza inclui arquivos de inspeção gerados pela ferramenta, mesmo em caso de erro.
with TemporaryDirectory(prefix="temporarios_", dir=arquivo.resolve().parent) as pasta_temporaria:
    temporario = Path(pasta_temporaria) / "base_verificacao.xlsx"
    pacote = Path(pasta_temporaria) / "operacao.json"
    matriz = [list(resultado.columns)] + json.loads(resultado.to_json(orient="values", force_ascii=False))
    pacote.write_text(json.dumps({
        "source": str(arquivo.resolve()), "target": str(temporario.resolve()),
        "originalRows": len(base_original) + 1, "values": matriz
    }, ensure_ascii=False), encoding="utf-8")
    runtime = Path(os.environ.get("USERPROFILE", str(Path.home()))) / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
    node = runtime / "bin/node.exe"
    ambiente = os.environ.copy()
    ambiente["CODEX_NODE_PACKAGES"] = str(runtime / "node_modules")
    codigo_js = "\nimport fs from 'node:fs/promises';\nimport {createRequire} from 'node:module';\nimport {pathToFileURL} from 'node:url';\nconst require = createRequire(process.env.CODEX_NODE_PACKAGES + '/../package.json');\nconst {FileBlob, SpreadsheetFile} = await import(pathToFileURL(require.resolve('@oai/artifact-tool')).href);\nconst job=JSON.parse(await fs.readFile(process.argv[1],'utf8'));\nconst wb=await SpreadsheetFile.importXlsx(await FileBlob.load(job.source));\nconst sheet=wb.worksheets.getItem('Sheet1');\nsheet.getRange('A1:H'+job.originalRows).clear({applyTo:'contents'});\nsheet.getRange('A1:H'+job.values.length).values=job.values;\nwb.recalculate();\nconsole.log((await wb.inspect({kind:'region',sheetId:'Sheet1',range:'A1:C5',maxChars:1000})).ndjson);\nconst out=await SpreadsheetFile.exportXlsx(wb);\nawait out.save(job.target);\n"
    processo = subprocess.run([str(node), "--input-type=module", "-e", codigo_js, str(pacote.resolve())],
                             env=ambiente, capture_output=True, text=True, encoding="utf-8")
    if processo.returncode:
        raise RuntimeError(processo.stderr or processo.stdout)
    verificada = pd.read_excel(temporario)
    pd.testing.assert_frame_equal(resultado, verificada, check_dtype=False)
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
    if not resultado.equals(base_original):
        os.replace(temporario, arquivo)
print("Correções aplicadas nesta execução:", len(alteracoes))
print("Remoções nesta execução:", quantidade_removida)
print("Total atual na base:", len(verificada))
print("Lemas e demais colunas das linhas mantidas: preservados.")


Correções aplicadas nesta execução: 0
Remoções nesta execução: 0
Total atual na base: 7774
Lemas e demais colunas das linhas mantidas: preservados.
